# Phase 7 v3 · Hard negatives and passage-pair reranking

Attach **phase2v3**, enable Internet and the HF download secret, and select **GPU T4 ×2** in Kaggle. The worker uses only GPU_INDEX = 1 by default. On a single-GPU runtime, set GPU_INDEX = 0. Keep v2 on GPU 0 if sharing a Kaggle session. Run all cells.

The model and tokenizer implementations come from the latest Phase 2 v3 notebook. The worker reads the architecture from final_model (1).pt and strictly loads its weights. It supports both attention variants in that notebook and targets the matching value projection.

## Pipeline

1. **Prepare.** Build split-specific BM25 rankings, tokenize with the original vocabulary, and retain up to eight 384-token windows per case. Long cases use evenly spaced windows over their full length. Preserve ID strings.
2. **Train the retriever.** Start LoRMA+ from the original base and train for one epoch using BM25 negatives. Rank all Train candidates, remove known positives and self-matches, save up to 20 hard negative IDs per query, then run one more epoch with a mix of mined and BM25 negatives. Keep the checkpoint with the best full-pool Dev Micro-F1@7.
3. **Train the reranker.** Start another LoRMA+ model from the same original base. Join the best-matching query and candidate passages into one 512-token input. Read a relevance logit from the last non-padding token. Train for up to one epoch on gold-vs-negative comparisons. Select by Dev Micro-F1@7.
4. **Export.** Rerank the union of the top 25 dense and top 25 BM25 results. Blend normalized reranker and retrieval scores at 70/30. Log Dev shortlist recall. Save both methods' Test predictions and select the stronger exported method by Dev F1.

The run keeps a 90-minute local deadline. The reranker estimates Test encoding time from a Dev pilot and reserves it before training.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import runpy
import shutil
import subprocess
import sys
import time

STARTED = time.monotonic()
DEADLINE = STARTED + 90 * 60
CHECKPOINT = Path("/kaggle/input/datasets/koooshiieee/phase2v3/final_model (1).pt")
LOCAL_DATA_ROOT = None  # Folder containing Train/, Dev/, Test/ if files are attached.
LOCAL_TOKENIZER_DIR = None  # Folder containing the original vocab.json and merges.json.
CHUNK_LENGTH = 384
WINDOW_STRIDE = 320
MAX_WINDOWS = 8
QUERY_BATCH = 2  # Use 1 if Kaggle reports CUDA OOM.
ENCODE_BATCH = 8  # Windows per forward pass.
GPU_INDEX = 1  # Use 0 on a single-GPU Kaggle session.
PAIR_LENGTH = 512
RERANK_TRAIN_BATCH = 4
RERANK_SCORE_BATCH = 12
RUN_ROOT = Path("/kaggle/working/phase7_v3") / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
RUN_ROOT.mkdir(parents=True)

## Materialize the original model, tokenizer, and retrieval workers

In [ ]:
source = r'''"""
Llama-style decoder-only LM, written from scratch (no HF / no llama library).

Pieces: RMSNorm, RoPE (or ALiBi), grouped-query attention (GQA; MQA = n_kv_heads 1,
MHA = n_kv_heads == n_heads), SwiGLU MLP, pre-norm residual blocks, tied embeddings,
KV-cache generation with left-padded batches.

Only torch tensors/autograd/nn.Linear/nn.Embedding are used as building blocks.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------- config
@dataclass
class Config:
    vocab_size: int = 33000
    d_model: int = 1024
    n_layers: int = 16
    n_heads: int = 16
    n_kv_heads: int = 4          # 16 query heads share 4 KV heads (4 queries per KV head)
    ffn_hidden: int = 2560
    max_seq_len: int = 1024      # training context
    rope_cache_len: int = 4096   # RoPE table length (>= prompt + generated tokens)
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    tie_embeddings: bool = True
    pos_enc: str = "rope"        # "rope" | "alibi"
    attn_type: str = "gqa"       # "gqa" | "mla"
    kv_latent: int = 128         # MLA only: size of the cached K/V latent
    attn_impl: str = "manual"    # "manual" (hand-written softmax(QK^T)V) | "sdpa" (torch fused kernel, same math)


PRESETS = {
    # exact totals with vocab 33000, tied embeddings
    "v2_100m": dict(d_model=768, n_layers=12, n_heads=12, n_kv_heads=4, ffn_hidden=2048), # ~100.86M
    "target_150m": dict(d_model=896, n_layers=14, n_heads=14, n_kv_heads=2, ffn_hidden=2304), # ~142M 
    "base":  dict(d_model=1024, n_layers=16, n_heads=16, n_kv_heads=4, ffn_hidden=2560),  # ~201.6M
    "small": dict(d_model=1024, n_layers=12, n_heads=16, n_kv_heads=4, ffn_hidden=2816),  # ~169.1M
    "v3_100m_mla": dict(d_model=768, n_layers=12, n_heads=12, n_kv_heads=12, ffn_hidden=1920, attn_type="mla", kv_latent=256),  # ~99.7M
    "v3_80m_mla": dict(d_model=640, n_layers=12, n_heads=10, n_kv_heads=10, ffn_hidden=1984, attn_type="mla", kv_latent=160),  # ~80.4M
    "v3_80m_gqa": dict(d_model=640, n_layers=12, n_heads=10, n_kv_heads=2, ffn_hidden=2048),  # ~80.1M, control
    "tiny":  dict(d_model=64,   n_layers=2,  n_heads=4,  n_kv_heads=2, ffn_hidden=128),   # smoke tests
}


def analytic_param_count(c: Config) -> int:
    """Closed-form parameter count (use it to size a model before building it)."""
    hd = c.d_model // c.n_heads
    if c.attn_type == "mla":
        attn = 2* c.d_model* c.n_heads*hd + c.d_model*c.kv_latent+c.kv_latent\
               + 2 * c.kv_latent * c.n_heads * hd        # Wq,Wo + W_dkv + kv_norm + W_uk,W_uv
    else:
        attn = 2 * c.d_model * c.n_heads * hd + 2 * c.d_model * c.n_kv_heads * hd
    mlp = 3 * c.d_model * c.ffn_hidden                                           # gate, up, down
    per_layer = attn + mlp + 2 * c.d_model                                       # + 2 RMSNorm
    total = per_layer * c.n_layers + c.vocab_size * c.d_model + c.d_model        # + embedding + final norm
    if not c.tie_embeddings:
        total += c.vocab_size * c.d_model
    return total


def count_params(model, verbose=True) -> int:
    """model.parameters() yields a tied (shared) tensor only once, so tied weights are counted once."""
    total = sum(p.numel() for p in model.parameters())
    if verbose:
        emb = model.tok_emb.weight.numel()
        blocks = sum(p.numel() for p in model.layers.parameters())
        print(f"total {total/1e6:.1f}M | transformer blocks {blocks/1e6:.1f}M | embedding {emb/1e6:.1f}M "
              f"({'tied with output head' if model.cfg.tie_embeddings else 'output head separate'}) "
              f"| analytic {analytic_param_count(model.cfg)/1e6:.1f}M")
    return total


# ----------------------------------------------------------------------------- RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()                                   # always normalise in fp32
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)


# ----------------------------------------------------------------------------- RoPE
def build_rope_cache(head_dim, max_len, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))   # (hd/2,)
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)                                                # (max_len, hd/2)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x: (B, H, T, hd); cos/sin: (T, hd/2). Rotates the two halves of each head vector (Llama layout)."""
    half = x.shape[-1] // 2
    xf = x.float()
    x1, x2 = xf[..., :half], xf[..., half:]
    cos = cos[None, None]
    sin = sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


# ----------------------------------------------------------------------------- ALiBi
def get_alibi_slopes(n_heads):
    def pow2(n):
        start = 2 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_heads).is_integer():
        s = pow2(n_heads)
    else:
        c = 2 ** math.floor(math.log2(n_heads))
        s = pow2(c) + pow2(2 * c)[0::2][: n_heads - c]
    return torch.tensor(s, dtype=torch.float32)


# ----------------------------------------------------------------------------- GQA
def repeat_kv(x, n_rep):
    """(B, n_kv, S, hd) -> (B, n_kv*n_rep, S, hd); query head h uses kv head h // n_rep."""
    if n_rep == 1:
        return x
    B, n_kv, S, hd = x.shape
    return x[:, :, None].expand(B, n_kv, n_rep, S, hd).reshape(B, n_kv * n_rep, S, hd)


class GQAttention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.nh, self.nkv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.n_rep = self.nh // self.nkv
        self.impl = cfg.attn_impl
        self.wq = nn.Linear(cfg.d_model, self.nh * self.hd, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wo = nn.Linear(self.nh * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)    # (B, nh,  T, hd)
        k = self.wk(x).view(B, T, self.nkv, self.hd).transpose(1, 2)   # (B, nkv, T, hd)
        v = self.wv(x).view(B, T, self.nkv, self.hd).transpose(1, 2)

        if cos is not None:                                            # RoPE on q and k (not v)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past is not None:                                           # cache holds only nkv heads
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        new_cache = (k, v) if use_cache else None
        S = k.shape[2]

        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # relative distance of every (query, key) pair; keys j <= query position are visible
        qpos = torch.arange(S - T, S, device=x.device)
        kpos = torch.arange(S, device=x.device)
        rel = qpos[:, None] - kpos[None, :]                            # (T, S)
        allowed = (rel >= 0)[None, None]                               # (1, 1, T, S)
        if key_valid is not None:                                      # hide left-padding keys
            allowed = allowed & key_valid[:, None, None, :]
        bias = None
        if alibi_slopes is not None:
            bias = -alibi_slopes.view(1, -1, 1, 1) * rel.clamp(min=0).float()[None, None]   # (1, nh, T, S)

        if self.impl == "sdpa" and key_valid is None and T == S:      # training fast path
            if bias is None:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            else:
                m = bias.masked_fill(~allowed, torch.finfo(q.dtype).min).to(q.dtype)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        else:                                                          # hand-written attention
            scores = ((q * (self.hd ** -0.5)) @ k.transpose(-2, -1)).float()   # (B, nh, T, S)
            if bias is not None:
                scores = scores + bias
            scores = scores.masked_fill(~allowed, -1e30)               # finite, so a fully-masked pad row can't give NaN
            probs = torch.softmax(scores, dim=-1).to(v.dtype)
            out = probs @ v                                            # (B, nh, T, hd)

        out = out.transpose(1, 2).reshape(B, T, self.nh * self.hd)
        return self.wo(out), new_cache

# ----------------------------------------------------------------------------- MLA
class MLAttention(nn.Module):
    """Simplified multi-head latent attention. K and V are rebuilt from one small latent per token;
    only the latent (B, S, kv_latent) is cached. ALiBi only."""
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        assert cfg.pos_enc == "alibi", "this MLA has no decoupled RoPE; use --pos_enc alibi"
        self.nh = cfg.n_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.impl = cfg.attn_impl
        self.wq = nn.Linear(cfg.d_model, self.nh * self.hd, bias=False)
        self.w_dkv = nn.Linear(cfg.d_model, cfg.kv_latent, bias=False)
        self.kv_norm = RMSNorm(cfg.kv_latent, cfg.norm_eps)
        self.w_uk = nn.Linear(cfg.kv_latent, self.nh * self.hd, bias=False)
        self.w_uv = nn.Linear(cfg.kv_latent, self.nh * self.hd, bias=False)
        self.wo = nn.Linear(self.nh * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        c = self.kv_norm(self.w_dkv(x))                               # (B, T, r)
        if past is not None:
            c = torch.cat([past, c], dim=1)                           # cache is just the latent
        new_cache = c if use_cache else None
        S = c.shape[1]
        k = self.w_uk(c).view(B, S, self.nh, self.hd).transpose(1, 2)
        v = self.w_uv(c).view(B, S, self.nh, self.hd).transpose(1, 2)

        qpos = torch.arange(S - T, S, device=x.device)
        kpos = torch.arange(S, device=x.device)
        rel = qpos[:, None] - kpos[None, :]
        allowed = (rel >= 0)[None, None]
        if key_valid is not None:
            allowed = allowed & key_valid[:, None, None, :]
        bias = None
        if alibi_slopes is not None:
            bias = -alibi_slopes.view(1, -1, 1, 1) * rel.clamp(min=0).float()[None, None]

        if self.impl == "sdpa" and key_valid is None and T == S:
            if bias is None:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            else:
                m = bias.masked_fill(~allowed, torch.finfo(q.dtype).min).to(q.dtype)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        else:
            scores = ((q * (self.hd ** -0.5)) @ k.transpose(-2, -1)).float()
            if bias is not None:
                scores = scores + bias
            scores = scores.masked_fill(~allowed, -1e30)
            probs = torch.softmax(scores, dim=-1).to(v.dtype)
            out = probs @ v

        out = out.transpose(1, 2).reshape(B, T, self.nh * self.hd)
        return self.wo(out), new_cache

# ----------------------------------------------------------------------------- SwiGLU
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden, bias=False)
        self.w_up = nn.Linear(d_model, hidden, bias=False)
        self.w_down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


# ----------------------------------------------------------------------------- block + model
class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = MLAttention(cfg) if cfg.attn_type == "mla" else GQAttention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = SwiGLU(cfg.d_model, cfg.ffn_hidden)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        h, new_cache = self.attn(self.attn_norm(x), cos, sin, alibi_slopes, key_valid, past, use_cache)
        x = x + h
        x = x + self.mlp(self.mlp_norm(x))
        return x, new_cache


class LlamaLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.pos_enc in ("rope", "alibi") and cfg.attn_impl in ("manual", "sdpa")
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

        if cfg.pos_enc == "rope":
            cos, sin = build_rope_cache(cfg.d_model // cfg.n_heads, cfg.rope_cache_len, cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
            self.alibi_slopes = None
        else:
            self.rope_cos = self.rope_sin = None
            self.register_buffer("alibi_slopes", get_alibi_slopes(cfg.n_heads), persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():       # GPT-2 style: shrink the residual-branch output projections
            if name.endswith("wo.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False, cache=None, start_pos=0,
                key_valid=None, all_logits=False):
        """
        training:  model(x, targets=y) -> scalar loss
        inference: model(idx, use_cache=True, cache=..., start_pos=..., key_valid=...) -> (logits, cache)
                   logits are for the last position only unless all_logits=True.
        """
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos = sin = None
        if self.cfg.pos_enc == "rope":
            assert start_pos + T <= self.cfg.rope_cache_len, "sequence longer than the RoPE table"
            cos = self.rope_cos[start_pos:start_pos + T]
            sin = self.rope_sin[start_pos:start_pos + T]

        new_caches = []
        for i, layer in enumerate(self.layers):
            past = cache[i] if cache is not None else None
            x, c = layer(x, cos, sin, self.alibi_slopes, key_valid, past, use_cache)
            new_caches.append(c)
        x = self.norm(x)

        if targets is not None:
            logits = self.lm_head(x)
            return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        logits = self.lm_head(x if all_logits else x[:, -1:, :])
        return logits, (new_caches if use_cache else None)


# ----------------------------------------------------------------------------- generation
@torch.no_grad()
def generate(model, prompts, max_new_tokens, eos_id, pad_id, temperature=0.1, top_k=30,
             top_p=0.95, rep_penalty=1.2, ban_eos=True):
    """
    prompts: list of lists of token ids (any lengths). Left-pads them into one batch, runs one prefill,
    then decodes with a KV cache. Returns a list of lists of generated ids (EOS / padding removed).
    temperature <= 0 -> greedy.
    """
    model.eval()
    device = next(model.parameters()).device
    V = model.cfg.vocab_size
    B = len(prompts)
    L = max(len(p) for p in prompts)
    ids = torch.full((B, L), pad_id, dtype=torch.long, device=device)
    key_valid = torch.zeros(B, L, dtype=torch.bool, device=device)
    seen = torch.zeros(B, V, dtype=torch.bool, device=device)
    for b, p in enumerate(prompts):
        t = torch.tensor(p, dtype=torch.long, device=device)
        ids[b, L - len(p):] = t
        key_valid[b, L - len(p):] = True
        seen[b, t] = True

    out = [[] for _ in range(B)]
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    amp = torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda"))
    with amp:
        logits, cache = model(ids, use_cache=True, key_valid=key_valid, start_pos=0)
        for step in range(max_new_tokens):
            lg = logits[:, -1, :].float()
            if ban_eos:
                lg[:, eos_id] = float("-inf")
            if rep_penalty != 1.0:                                     # CTRL-style repetition penalty
                pen = torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty)
                lg = torch.where(seen, pen, lg)
            if temperature <= 0:
                nxt = lg.argmax(-1)
            else:
                lg = lg / temperature
                if top_k and top_k > 0:
                    kth = torch.topk(lg, min(top_k, V)).values[:, -1:]
                    lg = lg.masked_fill(lg < kth, float("-inf"))
                if top_p < 1.0:
                    sl, si = torch.sort(lg, descending=True, dim=-1)
                    pr = torch.softmax(sl, dim=-1)
                    drop = (pr.cumsum(-1) - pr) > top_p                # keep the token that crosses top_p
                    sl = sl.masked_fill(drop, float("-inf"))
                    lg = torch.full_like(lg, float("-inf")).scatter(1, si, sl)
                nxt = torch.multinomial(torch.softmax(lg, dim=-1), 1).squeeze(1)

            nxt = torch.where(finished, torch.full_like(nxt, pad_id), nxt)
            for b in range(B):
                if not finished[b] and nxt[b].item() != eos_id:
                    out[b].append(int(nxt[b].item()))
            finished = finished | (nxt == eos_id)
            seen.scatter_(1, nxt[:, None], True)
            if bool(finished.all()):
                break
            key_valid = torch.cat([key_valid, torch.ones(B, 1, dtype=torch.bool, device=device)], dim=1)
            logits, cache = model(nxt[:, None], use_cache=True, cache=cache,
                                  start_pos=L + step, key_valid=key_valid)
    return out
'''
(RUN_ROOT / 'llama_scratch.py').write_text(source, encoding='utf-8')
print('Prepared llama_scratch.py')

In [ ]:
source = r'''#!/usr/bin/env python3
import os, json, regex
from typing import Dict, List, Tuple
from collections import Counter, defaultdict
from tqdm.auto import tqdm

import regex


_PAT = regex.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")


class PureByteLevelBPETokenizer:
    def __init__(self, vocab_size: int = 33000):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>", "<mask>"]

        self.encoder: Dict[bytes, int] = {}
        self.decoder: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], bytes] = {}       # pair -> merged token
        self.merge_ranks: Dict[Tuple[bytes, bytes], int] = {}    # pair -> order learned (lower = earlier)
        self._cache: Dict[str, List[int]] = {}

        self._init_base_vocab()

    def _init_base_vocab(self):
        for b in range(256):
            b_bytes = bytes([b])
            self.encoder[b_bytes] = b
            self.decoder[b] = b_bytes

        # special tokens
        for idx, token in enumerate(self.special_tokens, start=256):
            b_bytes = token.encode("utf-8")
            self.encoder[b_bytes] = idx
            self.decoder[idx] = b_bytes

    @property
    def pad_token_id(self) -> int:
        return self.encoder["<pad>".encode("utf-8")]

    @property
    def unk_token_id(self) -> int:
        return self.encoder["<unk>".encode("utf-8")]

    @property
    def bos_token_id(self) -> int:
        return self.encoder["<s>".encode("utf-8")]

    @property
    def eos_token_id(self) -> int:
        return self.encoder["</s>".encode("utf-8")]

    # ------------------------------------------------------------------ training
    @staticmethod
    def _merge_word(word, a, b, new):
        out, i, n = [], 0, len(word)
        while i < n:
            if i < n - 1 and word[i] == a and word[i + 1] == b:
                out.append(new)
                i += 2
            else:
                out.append(word[i])
                i += 1
        return out

    def train_from_texts(self, texts: List[str], target_vocab_size: int = None, min_word_freq: int = 2):
        if target_vocab_size is None:
            target_vocab_size = self.vocab_size
        num_merges = target_vocab_size - len(self.encoder)
        print(f"\nTraining byte level BPE from scratch. Target merges: {num_merges:,} | Base Vocab: {len(self.encoder)}")

        # 1) pre-tokenize + count unique words (the BPE then runs on this small table, not the raw text)
        word_counts = Counter()
        for text in tqdm(texts, desc="Counting pre-tokens"):
            if text and text.strip():
                word_counts.update(_PAT.findall(text))

        words, freqs = [], []
        for w, c in word_counts.items():
            if c >= min_word_freq:
                words.append([bytes([b]) for b in w.encode("utf-8", "replace")])
                freqs.append(c)
        del word_counts
        print(f"Unique words used for merge learning: {len(words):,}")

        # 2) initial pair statistics (updated incrementally afterwards)
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for i, w in enumerate(tqdm(words, desc="Initial pair counts")):
            for p in zip(w, w[1:]):
                pair_counts[p] += freqs[i]
                where[p].add(i)

        # 3) merge loop
        pbar = tqdm(total=num_merges, desc="Learning BPE Merges", dynamic_ncols=True)
        for _ in range(num_merges):
            if not pair_counts:
                print("No more candidate pairs found to merge.")
                break

            best_pair, freq = max(pair_counts.items(), key=lambda kv: kv[1])
            if freq < 2:
                print(f"Stopping early: Most frequent pair frequency is {freq}")
                break

            a, b = best_pair
            merged_token = a + b
            self.merges[best_pair] = merged_token
            self.merge_ranks[best_pair] = len(self.merge_ranks)
            if merged_token not in self.encoder:          # two merge paths can give the same bytes
                new_id = len(self.encoder)
                self.encoder[merged_token] = new_id
                self.decoder[new_id] = merged_token

            for i in list(where[best_pair]):
                w, f = words[i], freqs[i]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        del pair_counts[p]
                w = self._merge_word(w, a, b, merged_token)
                words[i] = w
                for p in zip(w, w[1:]):
                    pair_counts[p] += f
                    where[p].add(i)
            del where[best_pair]
            pair_counts.pop(best_pair, None)

            pbar.update(1)
            pbar.set_postfix({"vocab_size": len(self.encoder), "top_pair_freq": freq})

        pbar.close()
        self._cache.clear()
        print(f"Training finished! Final Vocab Size: {len(self.encoder):,}")

    # ------------------------------------------------------------------ encode / decode
    def _encode_word(self, word: str) -> List[int]:
        cached = self._cache.get(word)
        if cached is not None:
            return cached
        tokens = [bytes([b]) for b in word.encode("utf-8", "replace")]
        while len(tokens) >= 2:
            best_pair, best_rank = None, None
            for p in zip(tokens, tokens[1:]):
                r = self.merge_ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_pair, best_rank = p, r
            if best_pair is None:
                break
            tokens = self._merge_word(tokens, best_pair[0], best_pair[1], self.merges[best_pair])
        ids = [self.encoder.get(t, self.unk_token_id) for t in tokens]
        if len(self._cache) < 1_000_000:
            self._cache[word] = ids
        return ids

    def encode(self, text: str) -> List[int]:
        ids: List[int] = []
        for word in _PAT.findall(text):
            ids.extend(self._encode_word(word))
        return ids

    def decode(self, token_ids: List[int]) -> str:
        raw_bytes = b"".join(self.decoder.get(i, b"<unk>") for i in token_ids)
        return raw_bytes.decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ save / load
    def save(self, save_dir: str):
        os.makedirs(save_dir, exist_ok=True)
        vocab_path = os.path.join(save_dir, "vocab.json")
        merges_path = os.path.join(save_dir, "merges.json")
        config_path = os.path.join(save_dir, "tokenizer_config.json")

        # latin1 maps bytes 1:1 onto code points, so any byte string survives JSON
        ser_vocab = {k.decode("latin1"): v for k, v in self.encoder.items()}
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(ser_vocab, f, ensure_ascii=False, indent=2)

        # JSON (not a space-separated text file): tokens can contain spaces, newlines, \r, \xa0 ...
        ser_merges = [[p1.decode("latin1"), p2.decode("latin1")] for (p1, p2) in self.merges]  # in learned order
        with open(merges_path, "w", encoding="utf-8") as f:
            json.dump(ser_merges, f, ensure_ascii=False)

        config = {
            "tokenizer_class": "PureByteLevelBPETokenizer",
            "vocab_size": len(self.encoder),
            "special_tokens": self.special_tokens,
            "pad_token_id": self.pad_token_id,
            "bos_token_id": self.bos_token_id,
            "eos_token_id": self.eos_token_id
        }
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)

        print(f"Tokenizer saved successfully to {save_dir}")

    @classmethod
    def load(cls, save_dir: str) -> "PureByteLevelBPETokenizer":
        tok = cls()
        with open(os.path.join(save_dir, "vocab.json"), "r", encoding="utf-8") as f:
            ser_vocab = json.load(f)
        tok.encoder = {k.encode("latin1"): v for k, v in ser_vocab.items()}
        tok.decoder = {v: k.encode("latin1") for k, v in ser_vocab.items()}

        tok.merges, tok.merge_ranks = {}, {}
        with open(os.path.join(save_dir, "merges.json"), "r", encoding="utf-8") as f:
            for rank, (s1, s2) in enumerate(json.load(f)):
                p1, p2 = s1.encode("latin1"), s2.encode("latin1")
                tok.merges[(p1, p2)] = p1 + p2
                tok.merge_ranks[(p1, p2)] = rank
        return tok
'''
(RUN_ROOT / 'bpe_tokenizer.py').write_text(source, encoding='utf-8')
print('Prepared bpe_tokenizer.py')

In [ ]:
source = r'''from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
from time import perf_counter
import json
import math
import os
import random
import re
import shutil
import subprocess

started = perf_counter()
DATA_REPO = "Exploration-Lab/CS-781-Capstone"
DATA_REVISION = "main"  # Set to a commit hash for repeatable inputs.
LOCAL_DATA_ROOT = None  # Directory containing Train/, Dev/, and Test/.
CHECKPOINT = Path("/kaggle/input/datasets/koooshiieee/phase2v1/final_model.pt")
TOKENIZER_REPO = "DumbLad/legal-bpe-tokenizer-phase1-v2"
LOCAL_TOKENIZER_DIR = None  # Directory containing the original vocab.json and merges.json.
TOKEN_SAMPLE_PER_POOL = 32  # Set to 0 to skip tokenizer sampling.
BM25_DEV_QUERIES = 80  # Set to 0 to skip, or None to score all Dev queries.
SEED = 781
SPLITS = {"Train": "train", "Dev": "dev", "Test": "test"}

def hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
    except Exception:
        return None
    for name in ("HF", "HF_TOKEN"):
        try:
            token = secrets.get_secret(name)
        except Exception:
            continue
        if token:
            return token
    return None

def hub_file(repo, filename, repo_type, revision=None):
    try:
        from huggingface_hub import hf_hub_download
        from huggingface_hub.errors import GatedRepoError
    except ImportError as exc:
        raise RuntimeError("Install huggingface_hub or configure a local input path.") from exc

    token = hf_token()
    try:
        return Path(hf_hub_download(
            repo_id=repo, filename=filename, repo_type=repo_type, revision=revision, token=token
        ))
    except GatedRepoError as exc:
        raise RuntimeError(
            f"{repo} is gated. Enable the Kaggle secret HF or HF_TOKEN with a read token "
            "from a Hugging Face account that has access to the repo, then rerun. "
            "Alternatively, attach the files and configure their local input path."
        ) from exc
    except Exception as exc:
        raise RuntimeError(
            f"Could not fetch {repo}/{filename}. Check the Kaggle Internet setting, "
            "Hub access and the file path, or configure a local input path above."
        ) from exc

def data_path(folder, filename):
    relative = Path(folder) / filename
    if LOCAL_DATA_ROOT is not None:
        return Path(LOCAL_DATA_ROOT) / relative
    for root in (Path("Phase_7_PCR"), Path.cwd()):
        if (root / relative).is_file():
            return root / relative
    return hub_file(DATA_REPO, f"Phase_7_PCR/{relative.as_posix()}", "dataset", DATA_REVISION)

def unique_pairs(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key: {key!r}")
        result[key] = value
    return result

def reject_constant(value):
    raise ValueError(f"Invalid JSON constant: {value}")

def load_mapping(path):
    with path.open(encoding="utf-8") as stream:
        value = json.load(stream, object_pairs_hook=unique_pairs, parse_constant=reject_constant)
    if not isinstance(value, dict) or not value or any(not key.strip() for key in value):
        raise ValueError(f"{path}: expected a nonempty object keyed by case ID")
    return value

def sentences(record, case_id):
    text = record.get("text") if isinstance(record, dict) else record
    if isinstance(text, str):
        return [text]
    if isinstance(text, list) and all(isinstance(line, str) for line in text):
        return text
    raise ValueError(f"Case {case_id}: expected text as a string or list of sentences")

def percentiles(values):
    ordered = sorted(values)
    return {f"p{p}": ordered[round((len(ordered) - 1) * p / 100)] for p in (0, 50, 90, 95, 99, 100)}

WORD = re.compile(r"[^\W_]+")
'''
(RUN_ROOT / 'data_helpers.py').write_text(source, encoding='utf-8')
print('Prepared data_helpers.py')

In [ ]:
source = r'''"""Prepare split-local document tensors, BM25 scores and training groups."""
import argparse
from hashlib import sha256
from pathlib import Path
import time

import numpy as np
from scipy.sparse import csr_matrix
from collections import Counter
import torch

import data_helpers as source
from bpe_tokenizer import PureByteLevelBPETokenizer
from pcr_worker import atomic_json, metrics, prediction, rank_scores


def document_chunks(text, tokenizer, length):
    tokens = tokenizer.encode(text)
    width = length - 2
    last = max(0, len(tokens) - width)
    offsets = list(range(0, last + 1, WINDOW_STRIDE))
    if offsets[-1] != last:
        offsets.append(last)
    if len(offsets) > MAX_WINDOWS:
        indices = np.linspace(0, len(offsets) - 1, MAX_WINDOWS).round().astype(int)
        offsets = [offsets[index] for index in indices]
    result = np.full((MAX_WINDOWS, length), tokenizer.pad_token_id, dtype=np.int32)
    for row, offset in enumerate(offsets):
        window = [tokenizer.bos_token_id, *tokens[offset:offset + width], tokenizer.eos_token_id]
        result[row, :len(window)] = window
    return result

def terms(text):
    words = source.WORD.findall(text.casefold())
    return words if len(words) <= 4096 else words[:2048] + words[-2048:]


def bm25(queries, candidates):
    vocabulary = {}
    rows = [Counter(terms(text)) for text in candidates]
    for row in rows:
        for word in row:
            vocabulary.setdefault(word, len(vocabulary))

    def matrix(documents, binary=False):
        indices, values, offsets = [], [], [0]
        for row in documents:
            for word, frequency in row.items():
                if word in vocabulary:
                    indices.append(vocabulary[word])
                    values.append(1 if binary else frequency)
            offsets.append(len(indices))
        return csr_matrix((np.asarray(values, dtype=np.float32), indices, offsets),
                          shape=(len(documents), len(vocabulary)))

    counts = matrix(rows)
    query = matrix([Counter(terms(text)) for text in queries], binary=True)
    df = np.asarray(counts.getnnz(axis=0))
    idf = np.log1p((len(candidates) - df + .5) / (df + .5))
    size = np.asarray(counts.sum(axis=1)).ravel()
    norm = 1.2 * (.25 + .75 * size / max(float(size.mean()), 1))
    frequency = counts.data
    normalizer = np.repeat(norm, np.diff(counts.indptr))
    data = (2.2 * frequency * idf[counts.indices] / (frequency + normalizer)).astype(np.float32)
    weighted = csr_matrix((data, counts.indices, counts.indptr), shape=counts.shape)
    return (query @ weighted.T).toarray()


def prepare_split(folder, stem, tokenizer, length, cached_chunks):
    queries = source.load_mapping(source.data_path(folder, f"{stem}.json"))
    candidates = source.load_mapping(source.data_path(folder, f"{stem}_candidates.json"))
    qkeys, ckeys = sorted(queries), sorted(candidates)
    ccols = {key: i for i, key in enumerate(ckeys)}
    documents = []
    documents_by_id = {}

    def get_index(key, text):
        # Shared IDs with differing text keep their separate split-local representations.
        versions = documents_by_id.setdefault(key, [])
        for prior_text, index in versions:
            if prior_text == text:
                return index
        index = len(documents)
        documents.append(text)
        versions.append((text, index))
        return index

    qtext = ["\n".join(source.sentences(queries[key], key)) for key in qkeys]
    ctext = ["\n".join(source.sentences(candidates[key], key)) for key in ckeys]
    q_doc = np.array([get_index(key, text) for key, text in zip(qkeys, qtext)])
    c_doc = np.array([get_index(key, text) for key, text in zip(ckeys, ctext)])
    scores = bm25(qtext, ctext)
    same_case = np.equal.outer(qkeys, ckeys)
    if (len(ckeys) - same_case.sum(1) < 7).any():
        raise ValueError(f"{folder} lacks seven non-self candidates")
    gold = np.zeros_like(same_case)
    for row, key in enumerate(qkeys):
        record = queries[key]
        if folder == "Test":
            if "relevant_candidates" in record:
                raise ValueError("Unexpected Test labels")
            continue
        labels = record["relevant_candidates"]
        if not isinstance(labels, list):
            raise ValueError(f"Invalid gold labels for {key}")
        for label in labels:
            if isinstance(label, str) and label.strip():
                if label not in ccols or label == key or gold[row, ccols[label]]:
                    raise ValueError(f"Invalid, self, or duplicate gold link for {key}")
                gold[row, ccols[label]] = True

    hard = []
    if folder != "Test":
        order = np.argsort(-scores, axis=1, kind="stable")
        for row, cols in enumerate(order):
            eligible = ~(same_case[row, cols] | gold[row, cols])
            hard.append(cols[eligible][:24].copy())

    chunks = []
    for text in documents:
        # Pure tokenization cache keyed on the full text digest across splits.
        digest = sha256(text.encode("utf-8")).digest()
        if digest not in cached_chunks:
            cached_chunks[digest] = document_chunks(text, tokenizer, length)
        chunks.append(cached_chunks[digest])
    return dict(ids=torch.from_numpy(np.stack(chunks)), q_doc=q_doc, c_doc=c_doc,
                query_keys=qkeys, candidate_keys=ckeys, gold=gold, bm25=scores, hard=hard,
                self=same_case)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--root", required=True, type=Path)
    parser.add_argument("--data", type=Path)
    parser.add_argument("--tokenizer", type=Path)
    parser.add_argument("--length", type=int, default=768)
    parser.add_argument("--stride", type=int, default=640)
    parser.add_argument("--windows", type=int, default=12)
    args = parser.parse_args()
    if args.length < 16 or args.length > 1280:
        raise ValueError("Chunk size must be between 16 and 1280")
    if not 1 <= args.stride <= args.length - 2 or args.windows < 2:
        raise ValueError("Stride must cover the text and window cap must be at least two")
    global WINDOW_STRIDE, MAX_WINDOWS
    WINDOW_STRIDE, MAX_WINDOWS = args.stride, args.windows
    source.LOCAL_DATA_ROOT = args.data
    directory = args.tokenizer
    if directory is None:
        vocab = source.hub_file(source.TOKENIZER_REPO, "vocab.json", "model")
        merges = source.hub_file(source.TOKENIZER_REPO, "merges.json", "model")
        if vocab.parent != merges.parent:
            raise RuntimeError("Tokenizer files came from different revisions")
        directory = vocab.parent
    tokenizer = PureByteLevelBPETokenizer.load(str(directory))
    if len(tokenizer.encoder) != 33000 or max(tokenizer.encoder.values()) != 32999:
        raise ValueError("Expected the original 33,000-entry tokenizer")

    bundle = {"pad": tokenizer.pad_token_id, "bos": tokenizer.bos_token_id,
              "eos": tokenizer.eos_token_id}
    cache = {}
    for folder, stem in source.SPLITS.items():
        started = time.monotonic()
        split = prepare_split(folder, stem, tokenizer, args.length, cache)
        bundle[folder] = split
        print(f"{folder} · {len(split['query_keys'])} queries · {len(split['candidate_keys'])} candidates · "
              f"{len(split['ids'])} documents · {int(split['ids'][..., 0].ne(tokenizer.pad_token_id).sum()):,} windows · {time.monotonic() - started:.1f}s", flush=True)
    dev = bundle["Dev"]
    ranks = rank_scores(dev["bm25"], dev, 0.0)
    atomic_json(args.root / "bm25_dev.json", metrics(ranks, dev["gold"]))
    test = bundle["Test"]
    ranks = rank_scores(test["bm25"], test, 0.0)
    atomic_json(args.root / "bm25_prediction.json", prediction(ranks, test))
    temporary = args.root / "inputs.tmp"
    torch.save(bundle, temporary)
    temporary.replace(args.root / "inputs.pt")


if __name__ == "__main__":
    main()
'''
(RUN_ROOT / 'pcr_prepare.py').write_text(source, encoding='utf-8')
print('Prepared pcr_prepare.py')

In [ ]:
source = r'''"""Phase 7 retrieval training and export. Invoked by the Kaggle notebook."""
import argparse
import math
from contextlib import nullcontext
import json
from pathlib import Path
import time

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint as activation_checkpoint

from llama_scratch import Config, LlamaLM


class LowRank(nn.Module):
    def __init__(self, linear, multiplicative, rank=8):
        super().__init__()
        self.linear = linear
        self.multiplicative = multiplicative
        width = linear.out_features if multiplicative else linear.in_features
        self.down = nn.Linear(width, rank, bias=False)
        self.up = nn.Linear(rank, linear.out_features, bias=False)
        nn.init.zeros_(self.up.weight)

    def forward(self, x):
        y = self.linear(x)
        update = self.up(self.down(y if self.multiplicative else x))
        return y + update


class Retriever(nn.Module):
    def __init__(self, checkpoint, method, pad):
        super().__init__()
        cfg = Config(**checkpoint["cfg"])
        cfg.attn_impl = "sdpa"
        self.base = LlamaLM(cfg)
        self.base.load_state_dict(checkpoint["model"], strict=True)
        self.base.requires_grad_(False)
        self.pad = pad
        if method != "lorma":
            raise ValueError(f"Unexpected retrieval method: {method}")
        for block in self.base.layers:
            block.attn.wq = LowRank(block.attn.wq, multiplicative=True)
            name = "w_uv" if cfg.attn_type == "mla" else "wv"
            setattr(block.attn, name, LowRank(getattr(block.attn, name), multiplicative=True))

        self.head = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        nn.init.eye_(self.head.weight)

    def hidden_states(self, ids):
        length = ids.size(1)
        if length > self.base.cfg.max_seq_len:
            raise ValueError("Input exceeds the checkpoint's context length")
        cos = sin = None
        if self.base.cfg.pos_enc == "rope":
            cos = self.base.rope_cos[:length]
            sin = self.base.rope_sin[:length]
        slopes = self.base.alibi_slopes
        hidden = self.base.tok_emb(ids)
        for block in self.base.layers:
            if self.training:
                hidden, _ = activation_checkpoint(
                    block, hidden, cos, sin, slopes, None, None, False,
                    use_reentrant=False, preserve_rng_state=False,
                )
            else:
                hidden, _ = block(hidden, cos, sin, slopes, None, None, False)
        return self.base.norm(hidden)

    def window_embeddings(self, ids):
        valid = ids.ne(self.pad)
        hidden = self.hidden_states(ids)
        pooled = (hidden.float() * valid[..., None]).sum(1)
        pooled = pooled / valid.sum(1).clamp(min=1)[:, None]
        return F.normalize(self.head(pooled).float(), dim=-1)


def atomic_json(path, data):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(data, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    tmp.replace(path)


def atomic_weights(path, model):
    tmp = path.with_suffix(".tmp")
    weights = {name: p.detach().cpu() for name, p in model.named_parameters() if p.requires_grad}
    torch.save(weights, tmp)
    tmp.replace(path)


def cast_frozen(model, device):
    if device.type == "cuda":
        for parameter in model.parameters():
            if not parameter.requires_grad:
                parameter.data = parameter.data.half()


def build_model(checkpoint, method, pad, device):
    model = Retriever(checkpoint, method, pad)
    cast_frozen(model, device)
    return model.to(device)


def load_report(output):
    return json.loads((output / "dev.json").read_text(encoding="utf-8"))


def restore(model, path):
    weights = torch.load(path, map_location="cpu", weights_only=True)
    trainable = {name for name, p in model.named_parameters() if p.requires_grad}
    if set(weights) != trainable:
        raise ValueError(f"Unexpected checkpoint keys in {path}")
    model.load_state_dict(weights, strict=False)


def amp(device):
    return torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else nullcontext()


def normalize_rows(scores, excluded):
    finite = np.where(excluded, np.nan, scores)
    lo = np.nanmin(finite, axis=1, keepdims=True)
    hi = np.nanmax(finite, axis=1, keepdims=True)
    return (np.where(excluded, lo, scores) - lo) / np.maximum(hi - lo, 1e-8)


def rank_scores(dense, split, weight):
    dense = normalize_rows(dense, split["self"])
    lexical = normalize_rows(split["bm25"], split["self"])
    mixed = weight * dense + (1 - weight) * lexical
    mixed[split["self"]] = -np.inf
    return np.argsort(-mixed, axis=1, kind="stable")[:, :7]


def metrics(ranks, gold):
    tp = int(np.take_along_axis(gold, ranks, axis=1).sum())
    predicted = ranks.size
    relevant = int(gold.sum())
    return dict(f1=2 * tp / (predicted + relevant), precision=tp / predicted,
                recall=tp / max(1, relevant), tp=tp)


def sample_windows(ids, pad, rng, count=2):
    valid = ids[..., 0].ne(pad).numpy()
    indices = np.zeros((len(ids), count), dtype=np.int64)
    mask = np.zeros_like(indices, dtype=bool)
    for row, available in enumerate(valid):
        choices = np.flatnonzero(available)
        picks = rng.choice(choices, min(count, len(choices)), replace=False)
        indices[row, :len(picks)] = picks
        mask[row, :len(picks)] = True
    return indices, mask


def group_loss(scores, gold, excluded):
    negatives = ~(gold | excluded)
    pairs = gold[:, :, None] & negatives[:, None, :]
    # Each known positive is compared directly with the negatives for its query.
    deltas = (scores[:, None, :] - scores[:, :, None] + .05) / .10
    loss = F.softplus(deltas).masked_fill(~pairs, 0)
    counts = pairs.sum(dim=(1, 2))
    return ((loss.sum(dim=(1, 2)) / counts.clamp_min(1)).sum()
            / counts.gt(0).sum().clamp_min(1))


def passage_scores(q, c, q_valid, c_valid):
    cosine = torch.einsum("qxh,cyh->qcxy", q, c)
    valid = q_valid[:, None, :, None] & c_valid[None, :, None, :]
    cosine = cosine.masked_fill(~valid, -2.0)
    flat = cosine.flatten(2)
    first, index = flat.max(2)
    q_position = torch.arange(q.size(1), device=q.device)
    c_position = torch.arange(c.size(1), device=c.device)
    different_q = q_position[None, None, :, None] != (index // c.size(1))[:, :, None, None]
    different_c = c_position[None, None, None, :] != (index % c.size(1))[:, :, None, None]
    second = cosine.masked_fill(~(different_q & different_c), -2.0).flatten(2).amax(2)
    return torch.where(second > -1.5, (first + second) / 2, first)


def encode(model, ids, device, batch, deadline):
    valid = ids[..., 0].ne(model.pad)
    positions = valid.nonzero()
    result = torch.zeros((*valid.shape, model.base.cfg.d_model), device=device, dtype=torch.float16)
    model.eval()
    with torch.inference_mode(), amp(device):
        for offset in range(0, len(positions), batch):
            if time.monotonic() >= deadline:
                raise TimeoutError("Window encoding reached the stage deadline")
            doc, window = positions[offset:offset + batch].unbind(1)
            embeddings = model.window_embeddings(ids[doc, window].to(device))
            result[doc.to(device), window.to(device)] = embeddings.half()
    return result


def all_scores(vectors, valid, split, deadline):
    device = vectors.device
    queries = torch.as_tensor(split["q_doc"], device=device)
    candidates = torch.as_tensor(split["c_doc"], device=device)
    candidate_vectors = vectors[candidates]
    candidate_valid = valid[candidates]
    results = []
    for query in queries.split(1):
        if time.monotonic() >= deadline:
            raise TimeoutError("Passage scoring reached the stage deadline")
        results.append(passage_scores(vectors[query], candidate_vectors, valid[query], candidate_valid))
    return torch.cat(results).float().cpu().numpy()


def fixed_dev_loss(vectors, split, model, args):
    rng = np.random.default_rng(args.seed + 1)
    device = vectors.device
    indices, masks = sample_windows(split["ids"], model.pad, rng)
    rows = torch.arange(len(indices), device=device)[:, None]
    subset = vectors[rows, torch.from_numpy(indices).to(device)].float()
    mask = torch.from_numpy(masks).to(device)
    eligible = np.flatnonzero(split["gold"].any(1))
    losses = []
    for offset in range(0, len(eligible), args.query_batch):
        rows = eligible[offset:offset + args.query_batch]
        cols = choose_batch(split, rows, rng)
        q, c = split["q_doc"][rows], split["c_doc"][cols]
        scores = passage_scores(subset[q], subset[c], mask[q], mask[c])
        targets = torch.from_numpy(split["gold"][np.ix_(rows, cols)]).to(device)
        excluded = torch.from_numpy(split["self"][np.ix_(rows, cols)]).to(device)
        losses.append((float(group_loss(scores, targets, excluded)), len(rows)))
    return sum(loss * size for loss, size in losses) / sum(size for _, size in losses)


def validate(model, split, device, args):
    started = time.monotonic()
    with torch.inference_mode():
        vectors = encode(model, split["ids"], device, args.encode_batch, args.deadline)
        valid = split["ids"][..., 0].ne(model.pad).to(device)
        dense = all_scores(vectors, valid, split, args.deadline)
        loss = fixed_dev_loss(vectors, split, model, args)
    gold = split["gold"]
    results = [(metrics(rank_scores(dense, split, weight), gold), weight)
               for weight in (1.0, .75, .55, .35)]
    result, weight = max(results, key=lambda row: row[0]["f1"])
    return dict(**result, dense_f1=results[0][0]["f1"], dense_weight=weight,
                val_loss=loss, dev_seconds=time.monotonic() - started)


def update(model, split, rows, rng, device, optimizer, scaler):
    cols = choose_batch(split, rows, rng)
    docs = np.concatenate((split["q_doc"][rows], split["c_doc"][cols]))
    ids = split["ids"][docs]
    windows, valid = sample_windows(ids, model.pad, rng)
    ids = ids[torch.arange(len(ids))[:, None], torch.from_numpy(windows)].to(device)
    valid = torch.from_numpy(valid).to(device)
    mask = torch.from_numpy(split["self"][np.ix_(rows, cols)]).to(device)
    gold = torch.from_numpy(split["gold"][np.ix_(rows, cols)]).to(device)
    optimizer.zero_grad(set_to_none=True)
    with amp(device):
        embeddings = model.window_embeddings(ids.flatten(0, 1))
    embeddings = embeddings.reshape(*valid.shape, -1)
    count = len(rows)
    scores = passage_scores(embeddings[:count], embeddings[count:], valid[:count], valid[count:])
    loss = group_loss(scores, gold, mask)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
    scaler.step(optimizer)
    scaler.update()
    return float(loss.detach())

def choose_batch(split, rows, rng):
    candidates = []
    for row in rows:
        positives = np.flatnonzero(split["gold"][row])
        selected = rng.choice(positives, size=min(2, len(positives)), replace=False)
        lexical = split["hard"][row]
        if "mined" in split:
            dense = split["mined"][row]
            dense = rng.choice(dense, size=min(2, len(dense)), replace=False)
            lexical = lexical[~np.isin(lexical, dense)]
            hard = np.concatenate((dense, rng.choice(lexical, size=min(1, len(lexical)),
                                                      replace=False)))
        else:
            hard = rng.choice(lexical, size=min(3, len(lexical)), replace=False)
        candidates.extend(selected)
        candidates.extend(hard)
    return np.unique(candidates)


def mine(model, split, device, args, deadline):
    started = time.monotonic()
    with torch.inference_mode():
        vectors = encode(model, split["ids"], device, args.encode_batch, deadline)
        valid = split["ids"][..., 0].ne(model.pad).to(device)
        dense = all_scores(vectors, valid, split, deadline)
    output = {}
    mined = []
    for i, query in enumerate(split["query_keys"]):
        pool = np.flatnonzero(~(split["gold"][i] | split["self"][i]))
        order = pool[np.argsort(-dense[i, pool], kind="stable")[:20]]
        mined.append(order)
        output[query] = [split["candidate_keys"][j] for j in order]
    split["mined"] = mined
    atomic_json(args.root / "hard_negatives.json", output)
    print(f"Mined up to 20 dense negatives per Train query in "
          f"{time.monotonic()-started:.1f}s", flush=True)


def train(args, bundle, checkpoint, device):
    method = args.method
    output = args.root / method
    output.mkdir(exist_ok=True)
    started = time.monotonic()
    rng = np.random.default_rng(args.seed)
    model = build_model(checkpoint, method, bundle["pad"], device)
    train_data, dev_data = bundle["Train"], bundle["Dev"]
    initial = validate(model, dev_data, device, args)
    atomic_json(args.root / "initial_dev.json", initial)
    print(f"LoRMA+ initial · Dev pairwise loss {initial['val_loss']:.4f} · "
          f"F1@7 {initial['f1']:.4f} · Dev {initial['dev_seconds']:.1f}s", flush=True)
    initial_eval = initial["dev_seconds"]
    trainable = [p for p in model.parameters() if p.requires_grad]
    lr = 1.5e-4
    optimizer = torch.optim.AdamW(trainable, lr=lr, fused=device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    eligible = np.flatnonzero(train_data["gold"].any(1))
    if not len(eligible):
        raise ValueError("No training query has a positive")
    epoch = rng.permutation(eligible)
    offset = 0

    def next_rows():
        nonlocal epoch, offset
        if offset >= len(epoch):
            epoch = rng.permutation(eligible)
            offset = 0
        rows = epoch[offset:offset + args.query_batch]
        offset += len(rows)
        return rows

    train_count = len(eligible)
    min_queries = train_count
    max_queries = 2 * train_count
    validation_points = iter((min_queries, max_queries))
    next_check = next(validation_points)
    seen_queries = step = 0
    seconds_per_step = 0.0
    losses = []
    history = []
    best_key = best_file = None
    eval_reserve = max(1.0, initial_eval * 1.35) + 4.0
    train_until = args.deadline - eval_reserve
    print(f"Training on {train_count} labeled queries per epoch · "
          f"{len(train_data['query_keys']) - train_count} without positives · 2 epochs maximum",
          flush=True)

    while seen_queries < max_queries:
        if time.monotonic() + max(1.0, 1.5 * seconds_per_step) >= train_until:
            break
        rows = next_rows()
        tick = time.monotonic()
        model.train()
        fraction = seen_queries / max_queries
        decay = .15 + .85 * .5 * (1 + math.cos(math.pi * fraction))
        for group in optimizer.param_groups:
            group["lr"] = lr * min(1.0, (step + 1) / 100) * decay
        losses.append(update(model, train_data, rows, rng, device, optimizer, scaler))
        duration = time.monotonic() - tick
        seconds_per_step = 0.9 * seconds_per_step + 0.1 * duration if step else duration
        step += 1
        seen_queries += len(rows)

        now = time.monotonic()
        near_deadline = now + max(1.0, 1.5 * seconds_per_step) >= train_until
        if seen_queries < min_queries or (seen_queries < next_check and not near_deadline):
            continue
        if now + eval_reserve > args.deadline:
            break

        result = validate(model, dev_data, device, args)
        epoch_number = seen_queries / train_count
        train_loss = float(np.mean(losses))
        result.update(method=method, step=step, epochs=epoch_number, train_loss=train_loss,
                      trainable=sum(p.numel() for p in trainable),
                      parameters=sum(p.numel() for p in model.parameters()),
                      initial_f1=initial["f1"])
        history.append(result.copy())
        key = (result["f1"], -result["val_loss"])
        if best_key is None or key > best_key:
            new_file = output / f"checkpoint_{step}.pt"
            result["weights_file"] = new_file.name
            atomic_weights(new_file, model)
            atomic_json(output / "dev.json", result)
            if best_file is not None:
                best_file.unlink(missing_ok=True)
            best_key, best_file = key, new_file
        atomic_json(output / "history.json", history)
        while next_check <= seen_queries:
            next_check = next(validation_points, max_queries + 1)
        now = time.monotonic()
        estimated_steps = math.ceil((max_queries - seen_queries) / args.query_batch)
        eta = max(0, min(estimated_steps * seconds_per_step, train_until - now))
        print(f"{method} step {step} · epoch {epoch_number:.2f}/2 · "
              f"elapsed {now - started:.0f}s · ETA {eta:.0f}s · "
              f"train pairwise {train_loss:.4f} · Dev pairwise {result['val_loss']:.4f} · "
              f"F1@7 {result['f1']:.4f} · best {best_key[0]:.4f}", flush=True)
        losses.clear()
        if near_deadline:
            break
        if seen_queries == train_count:
            if time.monotonic() + max(20, 2 * initial_eval) >= train_until:
                break
            mine(model, train_data, device, args, train_until)

    atomic_json(output / "training.json", {
        "epochs_trained": seen_queries / train_count,
        "steps": step,
        "query_visits": seen_queries,
        "completed_two_epochs": seen_queries == max_queries,
        "mining_completed": "mined" in train_data,
        "qualified_checkpoint": best_key is not None,
    })
    if best_key is None:
        print(f"{method} INCOMPLETE · {seen_queries}/{min_queries} query visits "
              "or no completed Dev check · no qualifying checkpoint saved", flush=True)
    elif seen_queries < max_queries:
        print(f"{method} stopped at the time limit after {seen_queries / train_count:.2f} epochs",
              flush=True)


def prediction(ranks, split):
    result = {
        query: [split["candidate_keys"][int(i)] for i in row]
        for query, row in zip(split["query_keys"], ranks, strict=True)
    }
    pool = set(split["candidate_keys"])
    if set(result) != set(split["query_keys"]):
        raise ValueError("Prediction query IDs do not match the test split")
    if not all(len(ids) == len(set(ids)) == 7 and set(ids) <= pool for ids in result.values()):
        raise ValueError("Invalid ranked candidate IDs")
    return result


def export(args, bundle, checkpoint, device):
    for method in args.method.split(","):
        output = args.root / method
        report = load_report(output)
        model = build_model(checkpoint, method, bundle["pad"], device)
        restore(model, output / report["weights_file"])
        split = bundle["Test"]
        started = time.monotonic()
        with torch.inference_mode():
            vectors = encode(model, split["ids"], device, args.encode_batch, args.deadline)
            valid = split["ids"][..., 0].ne(model.pad).to(device)
            dense = all_scores(vectors, valid, split, args.deadline)
        ranks = rank_scores(dense, split, report["dense_weight"])
        atomic_json(output / "prediction.json", prediction(ranks, split))
        print(f"{method} · test export {time.monotonic() - started:.1f}s", flush=True)
        del model


def load_inputs(args):
    bundle = torch.load(args.root / "inputs.pt", weights_only=False, map_location="cpu")
    checkpoint = torch.load(args.checkpoint, weights_only=True, map_location="cpu")
    config = checkpoint["cfg"]
    if config.get("vocab_size") != 33000 or config.get("attn_type") not in ("mla", "gqa"):
        raise ValueError(f"Expected the original Phase 2 v3 checkpoint config: {config}")
    if bundle["Train"]["ids"].size(-1) > config["max_seq_len"]:
        raise ValueError("Configured passage length exceeds the Phase 2 v3 context")
    print(f"Phase 2 v3 · {config['attn_type']} · {config['pos_enc']} · "
          f"{config['n_layers']} layers · width {config['d_model']} · "
          f"context {config['max_seq_len']}", flush=True)
    return bundle, checkpoint


def main():
    p = argparse.ArgumentParser()
    p.add_argument("action", choices=("train", "export"))
    p.add_argument("method")
    p.add_argument("--root", type=Path, required=True)
    p.add_argument("--checkpoint", type=Path, required=True)
    p.add_argument("--deadline", type=float, required=True)
    p.add_argument("--encode-batch", type=int, default=16)
    p.add_argument("--query-batch", type=int, default=3)
    p.add_argument("--seed", type=int, default=781)
    args = p.parse_args()
    torch.set_num_threads(1)
    torch.manual_seed(args.seed)
    device = torch.device("cuda:0")
    if torch.cuda.device_count() != 1:
        raise RuntimeError("Expose exactly one CUDA GPU to the v3 worker")
    bundle, checkpoint = load_inputs(args)
    if args.action == "train":
        train(args, bundle, checkpoint, device)
    else:
        export(args, bundle, checkpoint, device)


if __name__ == "__main__":
    main()
'''
(RUN_ROOT / 'pcr_worker.py').write_text(source, encoding='utf-8')
print('Prepared pcr_worker.py')

In [ ]:
source = r'''"""Passage-pair reranking on the Phase 2 v3 checkpoint."""
import argparse
import json
import math
from pathlib import Path
import time

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

import pcr_worker as pcr


class CrossRanker(pcr.Retriever):
    def __init__(self, checkpoint, pad):
        super().__init__(checkpoint, "lorma", pad)
        self.head = nn.Linear(self.base.cfg.d_model, 1)

    def forward(self, ids):
        hidden = self.hidden_states(ids)
        last = ids.ne(self.pad).sum(1).clamp(min=1) - 1
        return self.head(hidden[torch.arange(len(ids), device=ids.device), last]).squeeze(-1).float()


def shortlist(dense, split, k=25):
    self_match = split["self"]
    dense_scores = np.where(self_match, -np.inf, dense)
    lexical_scores = np.where(self_match, -np.inf, split["bm25"])
    result = []
    for row, (d, b) in enumerate(zip(dense_scores, lexical_scores, strict=True)):
        eligible = np.count_nonzero(~self_match[row])
        count = min(k, eligible)
        first = np.argsort(-d, kind="stable")[:count]
        second = np.argsort(-b, kind="stable")[:count]
        result.append(np.union1d(first, second))
    return result


def passage_content(ids, pad, limit):
    content = ids[1:int(ids.ne(pad).sum()) - 1]
    begin = max(0, (len(content) - limit) // 2)
    return content[begin:begin + limit]


def make_pairs(split, vectors, rows, special, pair_length, deadline):
    ids = split["ids"]
    valid = ids[..., 0].ne(special["pad"]).to(vectors.device)
    result = []
    for row, cols in enumerate(rows):
        if time.monotonic() >= deadline:
            raise TimeoutError("Pair preparation reached its deadline")
        query = int(split["q_doc"][row])
        candidates = split["c_doc"][cols]
        if len(candidates) == 0:
            result.append(torch.empty((0, pair_length), dtype=ids.dtype))
            continue
        similarity = torch.einsum("xh,cyh->cxy", vectors[query], vectors[candidates])
        keep = valid[query][None, :, None] & valid[candidates][:, None, :]
        indices = similarity.masked_fill(~keep, -2).flatten(1).argmax(1).cpu().tolist()
        pairs = torch.full((len(cols), pair_length), special["pad"], dtype=ids.dtype)
        width = ids.shape[1]
        q_limit = (pair_length - 3) // 2
        c_limit = pair_length - 3 - q_limit
        for index, (candidate, match) in enumerate(zip(candidates, indices, strict=True)):
            q_index, c_index = divmod(match, width)
            q = passage_content(ids[query, q_index], special["pad"], q_limit)
            c = passage_content(ids[candidate, c_index], special["pad"], c_limit)
            pairs[index, 0] = special["bos"]
            pairs[index, 1:1+len(q)] = q
            pairs[index, 1+len(q)] = special["eos"]
            start = 2 + len(q)
            pairs[index, start:start+len(c)] = c
            pairs[index, start+len(c)] = special["eos"]
        result.append(pairs)
    return result


def train_rows(split, dense, seed, mined=None):
    candidate_columns = {case_id: col for col, case_id in enumerate(split["candidate_keys"])}
    rng = np.random.default_rng(seed)
    rows = []
    for i, gold in enumerate(split["gold"]):
        positives = np.flatnonzero(gold)
        if not len(positives):
            rows.append(np.empty(0, dtype=np.int64))
            continue
        allowed = ~(gold | split["self"][i])
        negatives = np.flatnonzero(allowed)
        if len(negatives) < 2:
            rows.append(np.empty(0, dtype=np.int64))
            continue
        if mined is None:
            dense_order = negatives[np.argsort(-dense[i, negatives], kind="stable")][:20]
        else:
            cols = [candidate_columns[key] for key in mined.get(split["query_keys"][i], [])
                    if key in candidate_columns and allowed[candidate_columns[key]]]
            dense_order = np.array(cols, dtype=np.int64)
            if not len(dense_order):
                dense_order = negatives[:20]
        bm25_order = negatives[np.argsort(-split["bm25"][i, negatives], kind="stable")][:20]
        pos = rng.choice(positives, size=min(2, len(positives)), replace=False)
        hard = int(rng.choice(dense_order))
        lexical = bm25_order[bm25_order != hard]
        if not len(lexical):
            lexical = negatives[negatives != hard]
        neg = [hard, int(rng.choice(lexical))]
        rows.append(np.array([value for j, negative in enumerate(neg)
                              for value in (pos[j % len(pos)], negative)]))
    return rows


def scores(model, pairs, device, batch, deadline):
    result = []
    model.eval()
    with torch.inference_mode(), pcr.amp(device):
        for chunk in pairs.split(batch):
            if time.monotonic() >= deadline:
                raise TimeoutError("Reranker scoring reached its deadline")
            result.append(model(chunk.to(device)).cpu())
    return torch.cat(result).numpy()


def unit(values):
    return (values - values.min()) / max(float(np.ptp(values)), 1e-8)


def ranked(scores_flat, rows, hybrid):
    top = []
    offset = 0
    for i, cols in enumerate(rows):
        neural = unit(scores_flat[offset:offset+len(cols)])
        fused = .7 * neural + .3 * unit(hybrid[i, cols])
        top.append(cols[np.argsort(-fused, kind="stable")[:7]])
        offset += len(cols)
    return np.stack(top)


def shortlist_recall(rows, gold):
    return sum(int(gold[i, cols].sum()) for i, cols in enumerate(rows)) / max(1, int(gold.sum()))


def prepare(args, bundle, checkpoint, device, report):
    retriever = pcr.build_model(checkpoint, "lorma", bundle["pad"], device)
    pcr.restore(retriever, args.root / "lorma" / report["weights_file"])
    result = {}
    mined_path = args.root / "hard_negatives.json"
    mined = json.loads(mined_path.read_text(encoding="utf-8")) if mined_path.is_file() else None
    for name in ("Test", "Dev", "Train"):
        start = time.monotonic()
        split = bundle[name]
        vectors = pcr.encode(retriever, split["ids"], device, args.encode_batch, args.deadline)
        valid = split["ids"][..., 0].ne(bundle["pad"]).to(device)
        dense = None if name == "Train" and mined is not None else pcr.all_scores(
            vectors, valid, split, args.deadline
        )
        weight = report["dense_weight"]
        hybrid = None
        if name != "Train":
            hybrid = weight * pcr.normalize_rows(dense, split["self"])
            hybrid += (1 - weight) * pcr.normalize_rows(split["bm25"], split["self"])
        if name == "Test":
            baseline = pcr.rank_scores(dense, split, weight)
            pcr.atomic_json(args.root / "lorma" / "prediction.json", pcr.prediction(baseline, split))
        rows = train_rows(split, dense, args.seed, mined) if name == "Train" else shortlist(dense, split)
        pair_list = make_pairs(split, vectors, rows, bundle, args.pair_length, args.deadline)
        result[name] = (torch.cat(pair_list), rows, hybrid)
        del vectors
        print(f"{name} · prepared {len(result[name][0]):,} reranker inputs in "
              f"{time.monotonic()-start:.1f}s", flush=True)
    del retriever
    return result


def run(args, bundle, checkpoint, device):
    retrieval = pcr.load_report(args.root / "lorma")
    prepared = prepare(args, bundle, checkpoint, device, retrieval)
    output = args.root / "reranker"
    output.mkdir(exist_ok=True)
    model = CrossRanker(checkpoint, bundle["pad"])
    pcr.cast_frozen(model, device)
    model.to(device)
    train = prepared["Train"][0].reshape(-1, 2, args.pair_length)
    dev_pairs, dev_rows, dev_hybrid = prepared["Dev"]
    test_pairs, test_rows, test_hybrid = prepared["Test"]

    started = time.monotonic()
    pilot_size = min(200, len(dev_pairs))
    scores(model, dev_pairs[:pilot_size], device, args.score_batch, args.deadline)
    per_pair = (time.monotonic() - started) / pilot_size
    eval_seconds = max(2, len(dev_pairs) * per_pair * 1.35)
    test_seconds = max(4, len(test_pairs) * per_pair * 1.35) + 20
    train_until = args.deadline - test_seconds
    recall = shortlist_recall(dev_rows, bundle["Dev"]["gold"])
    print(f"Dev shortlist recall {recall:.4f} · projected Dev {eval_seconds:.0f}s · "
          f"Test {test_seconds:.0f}s", flush=True)
    parameters = [p for p in model.parameters() if p.requires_grad]
    optim = torch.optim.AdamW(parameters, lr=1.5e-4, fused=device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    if len(train) == 0:
        raise ValueError("No Train comparisons could be built")
    order = np.random.default_rng(args.seed).permutation(len(train))
    best = None
    history = []
    seen = 0
    next_check = math.ceil(len(train) / 2)
    interval_losses = []
    seconds_per_step = 0
    for i in range(0, len(order), args.query_batch):
        if time.monotonic() + eval_seconds + max(2, 1.5 * seconds_per_step) >= train_until:
            break
        tick = time.monotonic()
        batch = train[order[i:i+args.query_batch]]
        model.train()
        optim.zero_grad(set_to_none=True)
        with pcr.amp(device):
            logits = model(batch.flatten(0, 1).to(device)).reshape(-1, 2)
        loss = F.softplus(logits[:, 1] - logits[:, 0]).mean()
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        nn.utils.clip_grad_norm_(parameters, 1.0)
        scaler.step(optim)
        scaler.update()
        duration = time.monotonic() - tick
        seconds_per_step = .9 * seconds_per_step + .1 * duration if seconds_per_step else duration
        seen += len(batch)
        interval_losses.append(float(loss.detach()))
        near_deadline = time.monotonic() + eval_seconds + max(2, 1.5 * seconds_per_step) >= train_until
        if seen < next_check and not near_deadline:
            continue
        if time.monotonic() + eval_seconds >= train_until and best is not None:
            break
        dev_scores = scores(model, dev_pairs, device, args.score_batch, train_until)
        ranks = ranked(dev_scores, dev_rows, dev_hybrid)
        result = pcr.metrics(ranks, bundle["Dev"]["gold"])
        result.update(method="reranker", pairs_trained=seen, shortlist_recall=recall,
                      train_loss=float(np.mean(interval_losses)))
        history.append(result.copy())
        if best is None or result["f1"] > best["f1"]:
            new_name = f"checkpoint_{seen}.pt"
            result["weights_file"] = new_name
            pcr.atomic_weights(output / new_name, model)
            pcr.atomic_json(output / "dev.json", result)
            if best:
                (output / best["weights_file"]).unlink(missing_ok=True)
            best = result
        pcr.atomic_json(output / "history.json", history)
        print(f"reranker {seen}/{len(train)} pair groups · loss {result['train_loss']:.4f} · "
              f"Dev F1@7 {result['f1']:.4f} · elapsed {time.monotonic()-started:.0f}s", flush=True)
        interval_losses.clear()
        next_check = len(train)
        if near_deadline:
            break

    if best is None:
        print("INCOMPLETE · no trained reranker checkpoint qualified for export", flush=True)
        return
    pcr.restore(model, output / best["weights_file"])
    test_scores = scores(model, test_pairs, device, args.score_batch, args.deadline)
    ranks = ranked(test_scores, test_rows, test_hybrid)
    pcr.atomic_json(output / "prediction.json", pcr.prediction(ranks, bundle["Test"]))
    print("Reranker Test predictions exported.", flush=True)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--root", type=Path, required=True)
    p.add_argument("--checkpoint", type=Path, required=True)
    p.add_argument("--deadline", type=float, required=True)
    p.add_argument("--encode-batch", type=int, default=8)
    p.add_argument("--query-batch", type=int, default=4)
    p.add_argument("--score-batch", type=int, default=16)
    p.add_argument("--pair-length", type=int, default=512)
    p.add_argument("--seed", type=int, default=781)
    args = p.parse_args()
    torch.set_num_threads(1)
    torch.manual_seed(args.seed)
    if torch.cuda.device_count() != 1:
        raise RuntimeError("Expose exactly one CUDA GPU to the v3 worker")
    bundle, checkpoint = pcr.load_inputs(args)
    if args.pair_length > checkpoint["cfg"]["max_seq_len"] or args.pair_length < 16:
        raise ValueError("Pair length exceeds the checkpoint context or is below 16")
    run(args, bundle, checkpoint, torch.device("cuda:0"))


if __name__ == "__main__":
    main()
'''
(RUN_ROOT / 'pcr_reranker.py').write_text(source, encoding='utf-8')
print('Prepared pcr_reranker.py')

## Run and export

Submit the top-level prediction.json, or use lorma/prediction.json and reranker/prediction.json separately. Each completed prediction contains seven unique Test candidates per query. summary.json identifies any unfinished stage and the selected method.

In [ ]:
def log(message):
    print(f"[{(time.monotonic() - STARTED) / 60:5.1f} min] {message}", flush=True)


def verify_runtime():
    import torch
    if not 0 <= GPU_INDEX < torch.cuda.device_count():
        raise RuntimeError(f"GPU_INDEX={GPU_INDEX} is unavailable. Enable a Kaggle GPU.")
    log(f"v3 device {GPU_INDEX} · {torch.cuda.get_device_name(GPU_INDEX)}")
    if CHECKPOINT.is_file():
        return CHECKPOINT
    matches = [path for path in Path("/kaggle/input").rglob("final_model (1).pt")
               if "phase2v3" in path.parts]
    if len(matches) != 1:
        raise FileNotFoundError("Attach the phase2v3 input or set CHECKPOINT to its final_model (1).pt.")
    return matches[0]


def run_stage(name, jobs, deadline):
    log(name)
    stage_dir = RUN_ROOT / "logs"
    stage_dir.mkdir(exist_ok=True)
    workers = []
    statuses = {}
    try:
        for label, command, gpu, overrides in jobs:
            path = stage_dir / f"{label}.log"
            env = os.environ.copy()
            if gpu is not None:
                env["CUDA_VISIBLE_DEVICES"] = str(gpu)
            env.update(OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", **overrides)
            with path.open("w", encoding="utf-8") as stream:
                process = subprocess.Popen(
                    [sys.executable, "-u", *map(str, command)],
                    cwd=RUN_ROOT, env=env, stdout=stream, stderr=subprocess.STDOUT,
                )
            workers.append((label, process, path.open(encoding="utf-8", errors="replace")))
        active = workers.copy()
        while active and time.monotonic() < deadline:
            for item in active.copy():
                label, process, reader = item
                lines = reader.read().strip()
                if lines:
                    print(lines, flush=True)
                if process.poll() is not None:
                    lines = reader.read().strip()
                    if lines:
                        print(lines, flush=True)
                    statuses[label] = process.returncode
                    active.remove(item)
            if active:
                time.sleep(.5)
    finally:
        for label, process, reader in workers:
            if process.poll() is None:
                process.kill()
                statuses[label] = "deadline_or_interrupt"
            else:
                statuses.setdefault(label, process.returncode)
            lines = reader.read().strip()
            if lines:
                print(lines, flush=True)
            reader.close()
    for label, status in statuses.items():
        if status != 0:
            log(f"{label} status {status} · see {stage_dir / (label + '.log')}")
    return statuses


def worker(action, method, deadline, checkpoint):
    return [
        "pcr_worker.py", action, method, "--root", RUN_ROOT, "--checkpoint", checkpoint,
        "--deadline", deadline, "--query-batch", QUERY_BATCH, "--encode-batch", ENCODE_BATCH]


def stage_end(minutes, reserve_minutes):
    return min(time.monotonic() + minutes * 60, DEADLINE - reserve_minutes * 60 - 15)


def run():
    checkpoint = verify_runtime()
    log(f"Phase 2 v3 base checkpoint · {checkpoint}")
    setup_command = ["pcr_prepare.py", "--root", RUN_ROOT, "--length", CHUNK_LENGTH,
                     "--stride", WINDOW_STRIDE, "--windows", MAX_WINDOWS]
    for flag, value in (("--data", LOCAL_DATA_ROOT), ("--tokenizer", LOCAL_TOKENIZER_DIR)):
        if value is not None:
            setup_command.extend((flag, value))

    setup_env = {}
    if LOCAL_DATA_ROOT is None or LOCAL_TOKENIZER_DIR is None:
        helpers = runpy.run_path(str(RUN_ROOT / "data_helpers.py"))
        token = helpers["hf_token"]()
        if token:
            setup_env["HF_TOKEN"] = token
    statuses = run_stage("Preparing PCR data and baseline", [("setup", setup_command, None, setup_env)],
                         stage_end(10, 75))
    if statuses.get("setup") != 0 or not (RUN_ROOT / "inputs.pt").is_file():
        raise RuntimeError("Setup did not finish. See the setup log above.")

    def trained(method):
        output = RUN_ROOT / method
        report = output / "dev.json"
        return report.is_file() and (output / json.loads(report.read_text())["weights_file"]).is_file()

    train_deadline = stage_end(43, 32)
    train_job = ("lorma", worker("train", "lorma", train_deadline - 3, checkpoint), GPU_INDEX, {})
    statuses.update(run_stage("LoRMA+ retriever · train, mine once, train again", [train_job], train_deadline))

    export_deadline = DEADLINE - 15
    if trained("lorma") and time.monotonic() + 30 < export_deadline:
        rerank_command = ["pcr_reranker.py", "--root", RUN_ROOT, "--checkpoint", checkpoint,
                          "--deadline", export_deadline - 3, "--encode-batch", ENCODE_BATCH,
                          "--query-batch", RERANK_TRAIN_BATCH, "--score-batch", RERANK_SCORE_BATCH,
                          "--pair-length", PAIR_LENGTH]
        rerank_job = ("reranker", rerank_command, GPU_INDEX, {})
        statuses.update(run_stage("Shortlist, train reranker, and export both methods", [rerank_job],
                                  export_deadline))

    baseline = RUN_ROOT / "lorma" / "prediction.json"
    if trained("lorma") and not baseline.is_file() and time.monotonic() + 30 < export_deadline:
        job = ("export_lorma", worker("export", "lorma", export_deadline - 3, checkpoint), GPU_INDEX, {})
        statuses.update(run_stage("Exporting retriever fallback", [job], export_deadline))

    methods = ("lorma", "reranker")
    reports = []
    for method in methods:
        report_path = RUN_ROOT / method / "dev.json"
        if report_path.is_file():
            report = json.loads(report_path.read_text(encoding="utf-8"))
            reports.append(report)
            print(f"{method:9} · Dev F1@7 {report['f1']:.4f} · "
                  f"precision {report['precision']:.4f} · recall {report['recall']:.4f}")
    reports.sort(key=lambda report: -report["f1"])
    exported = [report["method"] for report in reports
                if (RUN_ROOT / report["method"] / "prediction.json").is_file()]
    selected = exported[0] if exported else "bm25_fallback"
    selected_path = (RUN_ROOT / selected / "prediction.json"
                     if exported else RUN_ROOT / "bm25_prediction.json")
    target = RUN_ROOT / "prediction.json"
    shutil.copyfile(selected_path, target)
    training_file = RUN_ROOT / "lorma" / "training.json"
    training = json.loads(training_file.read_text(encoding="utf-8")) if training_file.is_file() else {}
    missing = [method for method in methods if method not in exported]
    summary = dict(selected=selected, completed=not missing and training.get("mining_completed", False)
                   and training.get("completed_two_epochs", False), missing=missing, reports=reports,
                   retriever_training=training, worker_statuses=statuses,
                   elapsed_seconds=time.monotonic() - STARTED)
    (RUN_ROOT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    if not summary["completed"]:
        log("INCOMPLETE · see summary.json for available predictions and completed stages")
    log(f"Selected {selected} · {target}")
    return summary


summary = run()

In [ ]:
UPLOAD_TO_HF = True  # Set True to upload after local export.
HF_REPO = "DumbLad/phase7-v3-pcr"

def upload_run():
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient

    if not (RUN_ROOT / "prediction.json").is_file():
        print("HF upload skipped · run training and export first.")
        return

    token = None
    try:
        secrets = UserSecretsClient()
        for name in ("HF_WRITE", "HF"):
            try:
                token = secrets.get_secret(name)
            except Exception:
                continue
            if token:
                break
    except Exception:
        pass
    if not token:
        print("HF upload skipped · enable HF or HF_WRITE for this Kaggle notebook. Local outputs are saved.")
        return

    try:
        api = HfApi(token=token)
        api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
        if not api.repo_info(HF_REPO, repo_type="model").private:
            print("HF upload skipped · destination repo must be private. Local outputs are saved.")
            return
        commit = api.upload_folder(
            repo_id=HF_REPO, repo_type="model", folder_path=RUN_ROOT,
            path_in_repo=f"runs/{RUN_ROOT.name}",
            allow_patterns=['prediction.json', 'summary.json', 'hard_negatives.json', 'bm25_prediction.json', '*_dev.json', '*/prediction.json', '*/dev.json', '*/training.json', '*/history.json', '*/checkpoint_*.pt', 'logs/*.log', '*.py'],
            commit_message=f"Upload Phase 7 v3 run {RUN_ROOT.name}",
        )
    except Exception as error:
        print(f"HF upload failed ({type(error).__name__}) · local outputs are saved. "
              "Check token write permissions and Internet, then retry this cell.")
        return
    print(f"Uploaded: {commit.commit_url}")

if UPLOAD_TO_HF:
    upload_run()
